In [2]:
#agentic RAG
# Agent自主决定何时检索、检索什么、用不用其他工具。
# 实现方式就是将检索器包装为Tool，Agent可以自主判合适调用工具来获取文档，甚至是多次检索，多轮迭代。

from langchain.agents.middleware import dynamic_prompt, ModelRequest
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()
from langchain.agents import create_agent
from langchain_core.vectorstores import InMemoryVectorStore
#VectorStore的用法
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage

#准备数据
markdown_content = open("resources/评估.md", "r").read()

#切分文本
headers_to_split = [
    ("#","Header1"),
    ("##","Header2"),
    ("###","Header3"),
]
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split,
    strip_headers=False # 不移除标题
    )

#切分文档, 返回的是 list[Document]
docs = markdown_splitter.split_text(markdown_content)

#给vectorstore添加id
for i, doc in enumerate(docs,start=1):
    doc.id = f"doc_{i}"

#初始化VectorStore
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_ollama import OllamaEmbeddings

#创建ollama模型
embeddings = OllamaEmbeddings(model="qwen3-embedding:0.6b",dimensions=1024)

#初始化向量数据库
vectorstore = InMemoryVectorStore(
    embedding=embeddings
)

# 保存文本到向量数据库,返回文档id
ids = vectorstore.add_documents(docs)




# ================Agentic RAG===================

# 1.加载环境变量
from dotenv import load_dotenv
load_dotenv()

#2.初始化模型
model = init_chat_model(
    model="deepseek-v4-flash",
    extra_body={"thinking": {"type": "disabled"}}
)

#3.定义工具
@tool
def retrieve_docs(query: str) -> str:
    """检索知识库，获取与用户提问相关的知识片段"""
    #1根据问题检索知识库
    docs = vectorstore.similarity_search(query)

    #2拼接知识片段
    content = "\n".join([doc.page_content for doc in docs])
    return content

#4.创建agent
agent = create_agent(
            model= model,
            tools= [retrieve_docs],
            system_prompt="你是一个专业的评估助手，能够根据用户的问题，只有问到与教资相关的问题才去调用检索工具，从知识库中检索相关知识片段，不能自己编造答案，答案要简短，直接引用原文。",
)

#5.测试
response =  await agent.ainvoke({"messages":[HumanMessage("论语中教育的目的是什么？")]})
for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

论语中教育的目的是什么？
================================== Ai Message ==================================

这个问题与教资无关，我不检索知识库。如果您有教资相关的问题，欢迎提问。
